<a href="https://colab.research.google.com/github/sunnykumaryadav1818-spec/2026b15210271-hcl-tech/blob/main/Project4_Air_Quality_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 4: Air Quality Insights Dashboard Dataset
**Industry:** Environment / Smart Cities  
**Dataset:** `04_air_quality.csv` (synthetic teaching dataset)  
**Tools:** Python, NumPy, Pandas (no charts – tables and printed results only)

**Business scenario:** Analyze air-quality readings across cities and identify periods and locations with higher particulate pollution.

> **Note:** The dataset is synthetic. Findings apply only to this data and do not prove causation.

## Step 0: Setup and load the file in Google Colab
Run the cell below. If `04_air_quality.csv` is not already in the Colab session, a file-upload box appears – choose the CSV file.

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

FILE_NAME = "04_air_quality.csv"

# In Google Colab: upload the CSV if it is not already present
if not os.path.exists(FILE_NAME):
    from google.colab import files
    uploaded = files.upload()          # choose 04_air_quality.csv
    FILE_NAME = list(uploaded.keys())[0]

print("Using file:", FILE_NAME)

## Task 1: Load and inspect the dataset

In [ ]:
df = pd.read_csv(FILE_NAME)

print("Shape (rows, columns):", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
display(df.head())
print("\nSummary statistics:")
display(df.describe())

### Duplicate check
Duplicate rows (same Date + City) would double-count a day, so they are checked and removed.

In [ ]:
dup_count = df.duplicated().sum()
print("Exact duplicate rows:", dup_count)
print("Duplicate Date+City combinations:", df.duplicated(subset=["Date", "City"]).sum())

df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

## Task 2: Identify missing values in all columns

In [ ]:
missing_table = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percent": (df.isnull().mean() * 100).round(2)
})
display(missing_table)

## Task 3: Handle missing PM2.5 values (with justification)

**Approach:** Fill each missing PM2.5 value with the **median PM2.5 of the same city and same month**.

**Why this method?**
- Pollution differs strongly between cities, so a single overall average would be misleading.
- Pollution also changes with season, so the month is taken into account.
- The median is used instead of the mean because PM2.5 has occasional very high spikes, which would pull a mean upward.
- Dropping rows would lose valid PM10 and temperature information.

PM10 and Temperature are filled the same way so later tasks can run on complete data.  
(The Date column is converted first because month is needed for the grouping.)

In [ ]:
# Date must be a datetime first (needed for the month used in the fill strategy)
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.month

def fill_with_group_median(data, column):
    """Fill NaN using city+month median, then city median as a fallback."""
    city_month_median = data.groupby(["City", "Month"])[column].transform("median")
    city_median = data.groupby("City")[column].transform("median")
    return data[column].fillna(city_month_median).fillna(city_median)

# keep a flag so we know which PM2.5 values were imputed
df["PM25_Imputed"] = df["PM2.5"].isnull()

for col in ["PM2.5", "PM10", "Temperature"]:
    df[col] = fill_with_group_median(df, col)

print("Missing values after cleaning:")
print(df[["PM2.5", "PM10", "Temperature"]].isnull().sum())
print("\nPM2.5 values that were imputed:", df["PM25_Imputed"].sum())

## Task 4: Convert Date to datetime and extract month

In [ ]:
# Date was already converted above; extract month name too for readable tables
df["Month_Name"] = df["Date"].dt.month_name()

print(df["Date"].dtype)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
display(df[["Date", "City", "Month", "Month_Name"]].head())

## Task 5: Average, median and maximum PM2.5

In [ ]:
pm25 = df["PM2.5"]
print(f"Average PM2.5 : {pm25.mean():.2f} µg/m³")
print(f"Median  PM2.5 : {pm25.median():.2f} µg/m³")
print(f"Maximum PM2.5 : {pm25.max():.2f} µg/m³")

# Where did the maximum occur?
max_row = df.loc[pm25.idxmax(), ["Date", "City", "PM2.5"]]
print("\nMaximum recorded on:")
print(max_row)

## Task 6: Rank cities by average PM2.5

In [ ]:
city_rank = (df.groupby("City")["PM2.5"]
               .agg(Average_PM25="mean", Median_PM25="median", Max_PM25="max", Readings="count")
               .round(2)
               .sort_values("Average_PM25", ascending=False))
city_rank["Rank"] = np.arange(1, len(city_rank) + 1)
display(city_rank)

print("Most polluted city (by average PM2.5):", city_rank.index[0])
print("Cleanest city (by average PM2.5):", city_rank.index[-1])

## Task 7: Dates with the highest PM2.5 readings

In [ ]:
top_dates = (df.sort_values("PM2.5", ascending=False)
               .loc[:, ["Date", "City", "PM2.5", "PM10", "Temperature"]]
               .head(10)
               .reset_index(drop=True))
top_dates.index = top_dates.index + 1
display(top_dates)

## Task 8: Compare PM10 and PM2.5 city-wise
PM2.5 is the fine fraction of PM10, so the **PM2.5 / PM10 ratio** shows how much of the particulate load is fine particles.

In [ ]:
pm_compare = df.groupby("City").agg(Avg_PM25=("PM2.5", "mean"), Avg_PM10=("PM10", "mean")).round(2)
pm_compare["PM25_to_PM10_Ratio"] = (pm_compare["Avg_PM25"] / pm_compare["Avg_PM10"]).round(3)
pm_compare["PM10_minus_PM25"] = (pm_compare["Avg_PM10"] - pm_compare["Avg_PM25"]).round(2)
display(pm_compare.sort_values("Avg_PM10", ascending=False))

## Task 9: Air-quality category column

**Thresholds used (documented assumption):** the instructor was to supply thresholds. Since none were included in the brief, the bands below are *assumed for teaching* and are **not claimed to be an official standard**.  
If your instructor gives different values, change only the `bins` list.

| PM2.5 (µg/m³) | Category |
|---|---|
| 0 – 30 | Good |
| 31 – 60 | Moderate |
| 61 – 90 | Unhealthy for Sensitive Groups |
| 91 – 120 | Unhealthy |
| 121 – 250 | Very Unhealthy |
| > 250 | Severe |

In [ ]:
bins = [0, 30, 60, 90, 120, 250, np.inf]
labels = ["Good", "Moderate", "Unhealthy (Sensitive)", "Unhealthy", "Very Unhealthy", "Severe"]

df["AQ_Category"] = pd.cut(df["PM2.5"], bins=bins, labels=labels, include_lowest=True)

print("Category counts:")
display(df["AQ_Category"].value_counts().reindex(labels).to_frame("Days"))

print("\nPercentage of readings in each category:")
display((df["AQ_Category"].value_counts(normalize=True).reindex(labels) * 100).round(2).to_frame("Percent"))

print("\nCategory share within each city (%):")
display((pd.crosstab(df["City"], df["AQ_Category"], normalize="index") * 100).round(1))

## Task 10: Monthly average PM2.5 by city

In [ ]:
monthly_city = (df.groupby(["City", "Month"])["PM2.5"].mean().round(2).reset_index())
display(monthly_city.head(12))

# Pivot: rows = month, columns = city (easier to read)
monthly_pivot = df.pivot_table(index="Month", columns="City", values="PM2.5", aggfunc="mean").round(2)
print("\nMonthly average PM2.5 (rows = month number, columns = city):")
display(monthly_pivot)

print("Most polluted month for each city:")
display(monthly_pivot.idxmax().to_frame("Worst_Month"))
print("Cleanest month for each city:")
display(monthly_pivot.idxmin().to_frame("Best_Month"))

## Task 11: NumPy standard deviation and percentiles

In [ ]:
pm25_array = df["PM2.5"].to_numpy()

print(f"Standard deviation of PM2.5 : {np.std(pm25_array):.2f}")
print(f"Variance of PM2.5           : {np.var(pm25_array):.2f}")

percentile_points = [10, 25, 50, 75, 90, 95, 99]
percentile_values = np.percentile(pm25_array, percentile_points)
for p, v in zip(percentile_points, percentile_values):
    print(f"{p:>3}th percentile: {v:8.2f}")

# City-wise standard deviation with NumPy (loop + function)
def city_std(data, city):
    return np.std(data.loc[data["City"] == city, "PM2.5"].to_numpy())

print("\nCity-wise standard deviation of PM2.5:")
for city in sorted(df["City"].unique()):
    print(f"  {city:<10}: {city_std(df, city):.2f}")

## Task 12: PM2.5 outliers using the IQR method
Outlier if value < Q1 − 1.5×IQR or value > Q3 + 1.5×IQR.

In [ ]:
q1, q3 = np.percentile(pm25_array, [25, 75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"Lower bound = {lower_bound:.2f}, Upper bound = {upper_bound:.2f}")

outliers = df[(df["PM2.5"] < lower_bound) | (df["PM2.5"] > upper_bound)]
print("\nNumber of outliers:", len(outliers))
print(f"Share of all readings: {len(outliers) / len(df) * 100:.2f}%")

print("\nOutliers by city:")
display(outliers["City"].value_counts().to_frame("Outlier_Count"))

print("Outlier list (top 15 by PM2.5):")
display(outliers.sort_values("PM2.5", ascending=False)[["Date", "City", "PM2.5", "PM10", "Temperature"]].head(15).reset_index(drop=True))

## Task 13: Compare temperature and PM2.5 using correlation
Correlation measures how two variables move together. It **does not prove** that one causes the other.

In [ ]:
overall_corr = df["Temperature"].corr(df["PM2.5"])
print(f"Overall correlation (Temperature vs PM2.5): {overall_corr:.3f}")

city_corr = (df.groupby("City")
               .apply(lambda g: g["Temperature"].corr(g["PM2.5"]), include_groups=False)
               .round(3).to_frame("Correlation"))
display(city_corr)

def describe_corr(r):
    strength = "strong" if abs(r) >= 0.6 else "moderate" if abs(r) >= 0.3 else "weak"
    direction = "negative" if r < 0 else "positive"
    return f"{strength} {direction}"

print("Interpretation (overall):", describe_corr(overall_corr), "relationship")
print("Note: this is an association in a synthetic dataset, not proof of causation.")
print("Overall correlation can also mix city differences; compare with the city-wise values above.")

## Task 14: Proportion of observations above a chosen PM2.5 threshold
**Chosen threshold:** 60 µg/m³ (an assumed teaching threshold, not an official standard). Change `THRESHOLD` to test others.

In [ ]:
THRESHOLD = 60

above = df["PM2.5"] > THRESHOLD
print(f"Readings above {THRESHOLD} µg/m³: {above.sum()} of {len(df)}")
print(f"Overall proportion: {above.mean():.4f}  ({above.mean() * 100:.2f}%)")

city_above = (df.assign(Above=above).groupby("City")["Above"].mean() * 100).round(2)
display(city_above.sort_values(ascending=False).to_frame(f"Percent_Above_{THRESHOLD}"))

month_above = (df.assign(Above=above).groupby("Month")["Above"].mean() * 100).round(2)
display(month_above.to_frame(f"Percent_Above_{THRESHOLD}").T)

## Task 15: Five data-supported public-awareness recommendations
The values in each recommendation are computed from the data (nothing is hard-coded).

In [ ]:
worst_city = city_rank.index[0]
best_city = city_rank.index[-1]
worst_city_avg = city_rank.loc[worst_city, "Average_PM25"]
best_city_avg = city_rank.loc[best_city, "Average_PM25"]

overall_monthly = df.groupby("Month")["PM2.5"].mean()
worst_month = int(overall_monthly.idxmax())
worst_month_name = pd.Timestamp(2024, worst_month, 1).month_name()
worst_month_avg = overall_monthly.max()

top_n_cities = city_above.sort_values(ascending=False).head(3)
highest_pct_city = top_n_cities.index[0]

outlier_months = outliers["Month"].value_counts()
spike_month = int(outlier_months.idxmax()) if len(outlier_months) else None

recommendations = [
    f"1. Target awareness campaigns in {worst_city}: it has the highest average PM2.5 "
    f"({worst_city_avg:.1f} µg/m³), about {worst_city_avg / best_city_avg:.1f}x that of {best_city} ({best_city_avg:.1f} µg/m³).",

    f"2. Issue advance seasonal advisories before {worst_month_name}, the month with the highest average PM2.5 "
    f"across cities ({worst_month_avg:.1f} µg/m³).",

    f"3. Use alerts based on the {THRESHOLD} µg/m³ assumed threshold: {above.mean() * 100:.1f}% of all readings exceed it, "
    f"and {highest_pct_city} exceeds it on {top_n_cities.iloc[0]:.1f}% of days.",

    f"4. Treat extreme-pollution days as early-warning events: {len(outliers)} readings ({len(outliers) / len(df) * 100:.1f}%) "
    f"are IQR outliers (above {upper_bound:.1f} µg/m³)" + (f", most often in month {spike_month}." if spike_month else "."),

    f"5. Advise sensitive groups (children, elderly, people with breathing problems) to limit outdoor exposure on days in the "
    f"'{labels[3]}' category or worse, which make up "
    f"{(df['AQ_Category'].isin(labels[3:])).mean() * 100:.1f}% of readings. "
    f"Temperature and PM2.5 show a {describe_corr(overall_corr)} association (r = {overall_corr:.2f}), "
    f"but this is only an association, not proof of cause.",
]

for r in recommendations:
    print(r, "\n")

## Limitations and assumptions
- The dataset is synthetic; real air-quality patterns can differ.
- Missing PM2.5 values were filled with the city-month median, which smooths extreme values.
- Air-quality categories and the 60 µg/m³ threshold are assumed for teaching, not official standards.
- Correlation shows association only, not causation.

## Save the cleaned dataset and summary tables

In [ ]:
cleaned = df.drop(columns=["PM25_Imputed"]).copy()
cleaned["Date"] = cleaned["Date"].dt.strftime("%Y-%m-%d")
cleaned.to_csv("04_air_quality_cleaned.csv", index=False)
city_rank.to_csv("city_ranking_pm25.csv")
monthly_pivot.to_csv("monthly_pm25_by_city.csv")
outliers[["Date", "City", "PM2.5", "PM10", "Temperature"]].to_csv("pm25_outliers.csv", index=False)

print("Saved: 04_air_quality_cleaned.csv, city_ranking_pm25.csv, monthly_pm25_by_city.csv, pm25_outliers.csv")

# Optional: download the cleaned file from Colab
# from google.colab import files
# files.download("04_air_quality_cleaned.csv")